# 🧬 Naïve Bayes sur de VRAIES molécules (ClinTox)

Nous allons lier nos deux approches ! Nous allons utiliser des **vraies molécules** (le dataset ClinTox des médicaments de la FDA), les transformer en vecteurs de 0 et de 1 (Morgan Fingerprints) grâce à RDKit/DeepChem, puis les donner à un algorithme classique **Naïve Bayes** de Scikit-Learn.

In [1]:
import logging
logging.getLogger("deepchem").setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')

import deepchem as dc
import numpy as np
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import classification_report, accuracy_score

## 1. Préparation des données (Featurizer)
Au lieu de prendre un CSV anonyme, on génère nous-mêmes les empreintes à 1024 bits avec `CircularFingerprint` (Morgan).

In [2]:
featurizer = dc.feat.CircularFingerprint(size=1024)

print("Chargement du dataset ClinTox...")
tasks, datasets, transformers = dc.molnet.load_clintox(featurizer=featurizer)
train_dataset, valid_dataset, test_dataset = datasets

# Le dataset ClinTox a 2 tâches : 0 = FDA_APPROVED, 1 = FDA_Tox
# On récupère uniquement la tâche 1 (Toxicité clinique)
X_train = train_dataset.X
y_train = train_dataset.y[:, 1] 

X_test = test_dataset.X
y_test = test_dataset.y[:, 1]

print(f"Taille de l'entraînement : {X_train.shape}")
print(f"Taille du test : {X_test.shape}")

Chargement du dataset ClinTox...


[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] Explicit valence for atom # 0 N, 4, is greater than permitted
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use MorganGenerator
[11:19:49] DEPRECATION WARNING: please use

Taille de l'entraînement : (1184, 1024)
Taille du test : (148, 1024)


## 2. Entraînement du modèle Naïve Bayes
Pour des données composées uniquement de 0 et de 1, la variante **BernoulliNB** est souvent beaucoup plus performante que le GaussianNB classique.

In [3]:
model = BernoulliNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("--- RÉSULTATS DU MODÈLE SUR LE JEU DE TEST ---")
print(classification_report(y_test, y_pred))

--- RÉSULTATS DU MODÈLE SUR LE JEU DE TEST ---
              precision    recall  f1-score   support

         0.0       0.95      0.96      0.95       138
         1.0       0.33      0.30      0.32        10

    accuracy                           0.91       148
   macro avg       0.64      0.63      0.63       148
weighted avg       0.91      0.91      0.91       148



## 3. Test sur nos molécules favorites
Le modèle est maintenant capable d'analyser n'importe quel SMILES !

In [4]:
smiles_a_tester = {
    "Paracétamol": "CC(=O)NC1=CC=C(O)C=C1",
    "Valdécoxib": "CC1=C(C(=NO1)C2=CC=CC=C2)C3=CC=C(C=C3)S(=O)(=O)N",
    "Cyanure": "C#N"
}

noms = list(smiles_a_tester.keys())
smiles = list(smiles_a_tester.values())

# 1. On transforme le texte (SMILES) en vecteur 0/1
X_nouvelles = featurizer.featurize(smiles)

# 2. On prédit
probabilites = model.predict_proba(X_nouvelles)

print("--- PRÉDICTIONS DE TOXICITÉ (Naïve Bayes) ---")
for i, nom in enumerate(noms):
    risque = probabilites[i][1] * 100
    if risque > 50:
        print(f"⚠️ {nom:12s} : TOXIQUE (Risque estimé à {risque:.1f}%)")
    else:
        print(f"✅ {nom:12s} : SÛR     (Risque estimé à {risque:.1f}%)")

--- PRÉDICTIONS DE TOXICITÉ (Naïve Bayes) ---
✅ Paracétamol  : SÛR     (Risque estimé à 0.0%)
✅ Valdécoxib   : SÛR     (Risque estimé à 0.0%)
✅ Cyanure      : SÛR     (Risque estimé à 0.0%)


[11:19:51] DEPRECATION WARNING: please use MorganGenerator
[11:19:51] DEPRECATION WARNING: please use MorganGenerator
[11:19:51] DEPRECATION WARNING: please use MorganGenerator


Ce résultat ridicule est en fait une excellente leçon. L'algorithme Naïve Bayes vient de faire exactement ce qu'on appelle en IA le "piège de la classe majoritaire" (ou la paresse de l'algorithme).

Voici pourquoi il a répondu "Sûr à 100%" pour un poison mortel comme le Cyanure :

1. Pourquoi le modèle est devenu paresseux ?
Le dataset ClinTox est extrêmement déséquilibré : il y a 90% de médicaments sûrs et 10% de médicaments toxiques. De plus, la machine Naïve Bayes est "Naïve". Elle ne comprend pas la chimie complexe, elle fait juste des statistiques basiques sur des colonnes. L'algorithme s'est dit : "Je vois que 90% des exemples qu'on m'a donnés sont SÛRS. Si je réponds que tout le monde est SÛR sans même regarder la molécule, je suis garanti d'avoir 90% de bonnes réponses à mon examen !"

C'est pour cela que dans le fichier Toxicite_Keras_2.ipynb, nous avions dû rajouter un code spécial (class_weights) pour "punir" la machine quand elle ignorait les molécules toxiques. Malheureusement, le vieil algorithme Naïve Bayes n'a pas cette fonctionnalité avancée de punition.

La conclusion : Naïve Bayes fonctionne très bien pour trier des e-mails (Spam / Non Spam), mais il est beaucoup trop simple pour faire de la découverte de médicaments. C'est exactement pour cela que le monde pharmaceutique a abandonné cet algorithme au profit des Réseaux de Neurones et des GNN !